# Option Valuation

This notebook covers option pricing using:
- **Binomial Tree** methods (European & American)
- **Black-Scholes** analytical formulas
- **Delta Hedging Simulation** using Euler-Maruyama discretization

---

## Setup & Parameters

Consider a European call option on a non-dividend-paying stock:
- **Maturity**: 1 year
- **Strike**: €99 (OTM call)
- **Risk-free rate**: 6%
- **Spot price**: €100
- **Volatility**: 20%

In [ ]:
import numpy as np
from scipy.stats import norm
import matplotlib.pyplot as plt

# Parameters
S0 = 100        # Initial stock price
K = 99          # Strike price
T = 1           # Time to maturity (years)
r = 0.06        # Risk-free rate
sigma = 0.2     # Volatility
N = 50          # Number of steps in binomial tree

---
## 1. Black-Scholes Model

The analytical Black-Scholes formula for a European call:

$$C = S_0 N(d_1) - K e^{-rT} N(d_2)$$

where:
$$d_1 = \frac{\ln(S_0/K) + (r + \sigma^2/2)T}{\sigma\sqrt{T}}, \quad d_2 = d_1 - \sigma\sqrt{T}$$

In [ ]:
def bs_call(s, k, t, rate, vol):
    """Black-Scholes European Call: returns (delta, price)."""
    if t <= 0:
        return (1.0 if s > k else 0.0), max(0, s - k)
    
    d1 = (np.log(s / k) + (rate + 0.5 * vol**2) * t) / (vol * np.sqrt(t))
    d2 = d1 - vol * np.sqrt(t)
    
    delta = norm.cdf(d1)
    price = s * delta - k * np.exp(-rate * t) * norm.cdf(d2)
    return delta, price

bs_delta, bs_price = bs_call(S0, K, T, r, sigma)
print(f"Black-Scholes Price: {bs_price:.4f}")
print(f"Black-Scholes Delta: {bs_delta:.4f}")

---
## 2. Binomial Tree Model

In [ ]:
def binomial_tree(s0, vol, t, n, rate):
    """Build stock price tree and return tree + parameters."""
    dt = t / n
    u = np.exp(vol * np.sqrt(dt))
    d = 1 / u
    p = (np.exp(rate * dt) - d) / (u - d)
    
    tree = np.zeros((n + 1, n + 1))
    for i in range(n + 1):
        for j in range(i + 1):
            tree[i, j] = s0 * (u ** j) * (d ** (i - j))
    
    return tree, u, d, p, dt


def european_call_value(tree, u, d, p, rate, k, dt, s0):
    """Price European call via backward induction. Returns (delta, price)."""
    n = tree.shape[0] - 1
    value = tree.copy()
    
    # Terminal payoffs
    for j in range(n + 1):
        value[n, j] = max(0, tree[n, j] - k)
    
    # Backward induction
    for i in reversed(range(n)):
        for j in range(i + 1):
            value[i, j] = np.exp(-rate * dt) * (p * value[i+1, j+1] + (1 - p) * value[i+1, j])
    
    delta = (value[1, 1] - value[1, 0]) / (s0 * (u - d))
    return delta, value[0, 0]

### 2.1: Binomial Tree Valuation

In [ ]:
tree, u, d, p, dt = binomial_tree(S0, sigma, T, N, r)
bt_delta, bt_price = european_call_value(tree, u, d, p, r, K, dt, S0)

print(f"Binomial Tree Price: {bt_price:.4f}")
print(f"Binomial Tree Delta: {bt_delta:.4f}")
print(f"\nDelta interpretation: {'OTM' if bt_delta < 0.5 else 'ATM' if bt_delta == 0.5 else 'ITM'}")

### 2.2: Binomial vs Black-Scholes Comparison

When volatility → 0 or → ∞, the difference between BT and BS tends to zero.

In [ ]:
n_values = [50, 100, 200]
vol_ranges = [np.linspace(0.01, 0.3, 50), np.linspace(0.5, 20, 50)]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for ax, vol_range in zip(axes, vol_ranges):
    for n_steps in n_values:
        diffs = []
        for vol in vol_range:
            _, bs_p = bs_call(S0, K, T, r, vol)
            tree, u, d, p, dt = binomial_tree(S0, vol, T, n_steps, r)
            _, bt_p = european_call_value(tree, u, d, p, r, K, dt, S0)
            diffs.append(bt_p - bs_p)
        ax.plot(vol_range, diffs, label=f"n={n_steps}")
    ax.set_xlabel("Volatility")
    ax.set_ylabel("Price Difference BT - BS")
    ax.legend()

plt.tight_layout()
plt.show()

### 2.3: Convergence Study

The binomial tree converges to Black-Scholes as $N \to \infty$.

In [ ]:
_, bs_ref = bs_call(S0, K, T, r, sigma)
steps_range = range(1, 101)
prices = []

for n_step in steps_range:
    tree, u, d, p, dt = binomial_tree(S0, sigma, T, n_step, r)
    _, price = european_call_value(tree, u, d, p, r, K, dt, S0)
    prices.append(price)

plt.figure(figsize=(10, 5))
plt.plot(steps_range, prices, label="Binomial Tree", alpha=0.8)
plt.axhline(bs_ref, color='r', linestyle='--', label="Black-Scholes")
plt.xlabel("Number of Steps (N)")
plt.ylabel("Option Price")
plt.title("2.3: Convergence of Binomial Tree to Black-Scholes")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

### 2.4: Delta Comparison

Compare the hedge parameter $\Delta$ from BT with $\Delta_{BS} = N(d_1)$.

In [ ]:
# First plot: Delta vs Volatility (single N value)
vol_ranges = [np.linspace(0.01, 0.3, 50), np.linspace(0.01, 20, 300)]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

n_steps = 50  # Only one N value

for ax, vol_range in zip(axes, vol_ranges):
    deltas = []
    for vol in vol_range:
        tree, u, d, p, dt = binomial_tree(S0, vol, T, n_steps, r)
        delta_0, _ = european_call_value(tree, u, d, p, r, K, dt, S0)
        deltas.append(delta_0)
    ax.plot(vol_range, deltas)
    ax.set_xlabel("Volatility")
    ax.set_ylabel("Delta (Binomial Tree)")

plt.tight_layout()
plt.show()

In [ ]:
# Second plot: Delta Difference (BT - BS) for different N values
n_values = [50, 100, 200]
vol_ranges = [np.linspace(0.01, 0.3, 50), np.linspace(0.5, 20, 50)]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for ax, vol_range in zip(axes, vol_ranges):
    for n_steps in n_values:
        diffs = []
        for vol in vol_range:
            # Binomial delta
            tree, u, d, p, dt = binomial_tree(S0, vol, T, n_steps, r)
            binomial_delta, _ = european_call_value(tree, u, d, p, r, K, dt, S0)
            
            # Black-Scholes delta
            bs_delta, _ = bs_call(S0, K, T, r, vol)
            
            # Store difference
            diffs.append(binomial_delta - bs_delta)
        
        ax.plot(vol_range, diffs, label=f"n={n_steps}")
    ax.set_xlabel("Volatility")
    ax.set_ylabel("Delta Difference (BT - BS)")
    ax.legend()

plt.tight_layout()
plt.show()

---
## 3. American Options

American options allow early exercise. We modify backward induction to compare continuation value vs immediate exercise.

In [ ]:
def american_option(price_tree, t, rate, k, vol, n, option_type="Call"):
    """Price American option with early exercise."""
    tree = price_tree.copy()
    dt = t / n
    u = np.exp(vol * np.sqrt(dt))
    d = 1 / u
    p = (np.exp(rate * dt) - d) / (u - d)
    
    rows, cols = tree.shape
    
    # Terminal payoffs
    for j in range(cols):
        spot = price_tree[rows - 1, j]
        if option_type == "Call":
            tree[rows - 1, j] = max(0, spot - k)
        else:
            tree[rows - 1, j] = max(0, k - spot)
    
    # Backward induction with early exercise
    for i in range(rows - 2, -1, -1):
        for j in range(i + 1):
            cont_val = np.exp(-rate * dt) * (p * tree[i+1, j+1] + (1 - p) * tree[i+1, j])
            spot = price_tree[i, j]
            exercise = max(0, spot - k) if option_type == "Call" else max(0, k - spot)
            tree[i, j] = max(exercise, cont_val)
    
    return tree[0, 0]

In [ ]:
tree, _, _, _, _ = binomial_tree(S0, sigma, T, N, r)

am_call = american_option(tree, T, r, K, sigma, N, "Call")
am_put = american_option(tree, T, r, K, sigma, N, "Put")

print(f"American Call: {am_call:.4f}")
print(f"American Put:  {am_put:.4f}")

We can see from the results that the American call option has the same value as the European one. That is because it is never optimal to exercise early an American call option on a non-dividend-paying stock. 

In [ ]:
n_steps = 50
vol_ranges = [np.linspace(0.01, 0.08, 50), np.linspace(0.2, 2.0, 50)]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for ax, vol_range in zip(axes, vol_ranges):
    call_prices = []
    put_prices = []
    for vol in vol_range:
        tree, _, _, _, _ = binomial_tree(S0, vol, T, n_steps, r)
        
        call_price = american_option(tree.copy(), T, r, K, vol, n_steps, "Call")
        put_price = american_option(tree.copy(), T, r, K, vol, n_steps, "Put")
        
        call_prices.append(call_price)
        put_prices.append(put_price)
    
    ax.plot(vol_range, call_prices, label="Call")
    ax.plot(vol_range, put_prices, '--', label="Put")
    ax.set_xlabel("Volatility")
    ax.set_ylabel("Option Price")
    ax.legend()

plt.tight_layout()
plt.show()

---
## 4. Delta Hedging Simulation

Use the Euler method to simulate stock paths and perform discrete delta hedging.

**Stock dynamics** (Euler-Maruyama):
$$S_{t+\Delta t} = S_t \left(1 + r\Delta t + \sigma \sqrt{\Delta t} Z\right), \quad Z \sim N(0,1)$$

**Experiments**:
1. Match volatility in stock process with volatility used for delta computation
2. Vary hedge frequency (daily → weekly)
3. Volatility mismatch scenarios

In [ ]:
def euler_path(s0, rate, vol, t, steps):
    """Generate stock path using Euler-Maruyama discretization."""
    dt = t / steps
    path = np.zeros(steps + 1)
    path[0] = s0
    
    # Vectorized random draws for efficiency
    z = np.random.normal(0, 1, steps)
    for i in range(1, steps + 1):
        path[i] = path[i-1] * (1 + rate * dt + vol * np.sqrt(dt) * z[i-1])
    
    return path


def hedging_error(s0, k, t, rate, sigma_stock, sigma_hedge, steps, hedge_interval=1):
    """
    Compute hedging error for a single path.
    
    Args:
        sigma_stock: Actual volatility driving the stock
        sigma_hedge: Volatility used in delta calculation (model assumption)
        hedge_interval: Rebalance every N steps (1=daily, 5=weekly for 252 steps/year)
    
    Returns:
        Hedging error (portfolio value - option payoff at maturity)
    """
    dt = t / steps
    path = euler_path(s0, rate, sigma_stock, t, steps)
    
    # Initial position
    t_remaining = t
    delta, option_price = bs_call(s0, k, t_remaining, rate, sigma_hedge)
    cash = option_price - delta * s0  # Borrow to buy delta shares
    
    # Rebalancing loop
    for i in range(1, steps + 1):
        # Accrue interest on cash (can be negative = borrowing cost)
        cash *= np.exp(rate * dt)
        
        # Rebalance at specified intervals or at final step
        if i % hedge_interval == 0 or i == steps:
            t_remaining = t - i * dt
            spot = path[i]
            
            if t_remaining <= 0:
                new_delta = 1.0 if spot > k else 0.0
            else:
                new_delta, _ = bs_call(spot, k, t_remaining, rate, sigma_hedge)
            
            # Adjust position: buy/sell shares
            cash -= (new_delta - delta) * spot
            delta = new_delta
    
    # Final portfolio value vs payoff
    portfolio_value = cash + delta * path[-1]
    payoff = max(0, path[-1] - k)
    
    return portfolio_value - payoff

In [ ]:
# Experiment 1: Matching volatilities, varying hedge frequency
np.random.seed(42)

M = 252  # Daily steps per year
n_sims = 500
frequencies = [1, 5, 10, 21]  # Daily, weekly, bi-weekly, monthly
labels = ['Daily', 'Weekly', 'Bi-weekly', 'Monthly']

print("Experiment 1: Varying Hedge Frequency (sigma_stock = sigma_hedge = 20%)\n")
print(f"{'Frequency':<12} {'Mean Error':>12} {'Std Dev':>12} {'RMSE':>12}")
print("-" * 50)

freq_results = {}
for freq, label in zip(frequencies, labels):
    errors = [hedging_error(S0, K, T, r, sigma, sigma, M, freq) for _ in range(n_sims)]
    errors = np.array(errors)
    freq_results[label] = errors
    rmse = np.sqrt(np.mean(errors**2))
    print(f"{label:<12} {errors.mean():>12.4f} {errors.std():>12.4f} {rmse:>12.4f}")

### Hedge Frequency Impact

More frequent rebalancing → lower hedging error (closer to continuous hedging assumption of Black-Scholes).

In [ ]:
# Visualize hedging error distribution by frequency
fig, axes = plt.subplots(1, 4, figsize=(14, 3), sharey=True)

for ax, label in zip(axes, labels):
    ax.hist(freq_results[label], bins=30, edgecolor='black', alpha=0.7)
    ax.axvline(0, color='r', linestyle='--')
    ax.set_xlabel("Hedging Error")
    ax.set_title(label)

axes[0].set_ylabel("Frequency")
plt.suptitle("Hedging Error Distribution by Rebalancing Frequency")
plt.tight_layout()
plt.show()

Having assumed that the volatility in the stock process matches the volatility used in the delta computation (equals to 20\%), when hedging is frequent, hedging-error distribution is centered very close to 0 with small spread. But the less the frequent rebalancing, the more large the variance, which means larger RMSE (so the distribution widens).

### Volatility Mismatch Experiments

What happens when the volatility used for delta computation differs from actual market volatility?

- **Underestimate**: sigma_hedge < sigma_stock → systematic under-hedging
- **Overestimate**: sigma_hedge > sigma_stock → systematic over-hedging

In [ ]:
# Experiment 2: Volatility mismatch (weekly hedging)
np.random.seed(123)

sigma_hedge_fixed = 0.20
sigma_stock_values = [0.10, 0.15, 0.20, 0.25, 0.30]
hedge_freq = 5  # Weekly

print("Experiment 2: Volatility Mismatch (Weekly Hedging, sigma_hedge = 20%)\n")
print(f"{'sigma_stock':<12} {'Mean Error':>12} {'Std Dev':>12} {'RMSE':>12}")
print("-" * 55)

vol_results = {}
for sigma_stock in sigma_stock_values:
    errors = [hedging_error(S0, K, T, r, sigma_stock, sigma_hedge_fixed, M, hedge_freq) 
              for _ in range(n_sims)]
    errors = np.array(errors)
    vol_results[sigma_stock] = errors
    rmse = np.sqrt(np.mean(errors**2))
    print(f"{sigma_stock:<12.2f} {errors.mean():>12.4f} {errors.std():>12.4f} {rmse:>12.4f}")

In [ ]:
# Visualize volatility mismatch impact
fig, axes = plt.subplots(1, 5, figsize=(16, 3), sharey=True)

labels_vol = ['sigma=10% (over)', 'sigma=15% (over)', 'sigma=20% (match)', 'sigma=25% (under)', 'sigma=30% (under)']
for ax, sigma_stock, label in zip(axes, sigma_stock_values, labels_vol):
    ax.hist(vol_results[sigma_stock], bins=30, edgecolor='black', alpha=0.7)
    ax.axvline(0, color='r', linestyle='--')
    ax.set_xlabel("Hedging Error")
    ax.set_title(label)

axes[0].set_ylabel("Frequency")
plt.suptitle("Hedging Error: Volatility Mismatch (sigma_hedge = 20%)")
plt.tight_layout()
plt.show()

It is evident from the distribution plots that volatility mismatch produces a lot of bias. So, we end up with either systematic under-hedging, which means the mean hedging error is negative (portfolio underperforms payoff), or systematic over-hedging, which means the mean hedging error is positive. 

In [ ]:
# Single path visualization: consolidated 2x2 comparison
np.random.seed(42)

# Generate one stock path to use for both daily and weekly comparison
stock_path = euler_path(S0, r, sigma, T, M)

def simulate_portfolio_with_path(stock_path, k, t, rate, sigma_hedge, hedge_interval=1):
    """Track portfolio and option values over time using a given stock path."""
    steps = len(stock_path) - 1
    dt = t / steps

    portfolio = np.zeros(steps + 1)
    option_val = np.zeros(steps + 1)
    deltas = np.zeros(steps + 1)

    # Initial setup
    delta, opt_price = bs_call(stock_path[0], k, t, rate, sigma_hedge)
    cash = opt_price - delta * stock_path[0]

    portfolio[0] = opt_price
    option_val[0] = opt_price
    deltas[0] = delta

    for i in range(1, steps + 1):
        cash *= np.exp(rate * dt)
        t_remaining = t - i * dt
        spot = stock_path[i]

        # Calculate current option value
        if t_remaining <= 0:
            option_val[i] = max(0, spot - k)
        else:
            _, option_val[i] = bs_call(spot, k, t_remaining, rate, sigma_hedge)

        # Rebalance at intervals
        if i % hedge_interval == 0 or i == steps:
            if t_remaining <= 0:
                new_delta = 1.0 if spot > k else 0.0
            else:
                new_delta, _ = bs_call(spot, k, t_remaining, rate, sigma_hedge)
            cash -= (new_delta - delta) * spot
            delta = new_delta

        portfolio[i] = cash + delta * spot
        deltas[i] = delta

    return portfolio, option_val, deltas

# Compare daily vs weekly hedging on the same path
port_daily, opt_val, deltas_daily = simulate_portfolio_with_path(stock_path, K, T, r, sigma, 1)
port_weekly, _, deltas_weekly = simulate_portfolio_with_path(stock_path, K, T, r, sigma, 5)

times = np.linspace(0, T, M + 1)

# Consolidated plot: Stock, Delta, Portfolio vs Option Value
fig, axes = plt.subplots(2, 2, figsize=(12, 8))

# Stock path
axes[0, 0].plot(times, stock_path)
axes[0, 0].axhline(K, color='r', linestyle='--', alpha=0.5, label='Strike')
axes[0, 0].set_ylabel('Stock Price')
axes[0, 0].set_title('Stock Price Path')
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

# Delta comparison
axes[0, 1].plot(times, deltas_daily, 'g-', label='Daily')
axes[0, 1].plot(times, deltas_weekly, 'r--', label='Weekly')
axes[0, 1].set_ylabel('Delta')
axes[0, 1].set_title('Delta Over Time')
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.3)

# Portfolio vs Option (Daily)
axes[1, 0].plot(times, port_daily, label='Portfolio')
axes[1, 0].plot(times, opt_val, '--', alpha=0.7, label='Option Value')
axes[1, 0].set_xlabel('Time (Years)')
axes[1, 0].set_ylabel('Value')
axes[1, 0].set_title(f"Daily Hedging (Error: {port_daily[-1] - opt_val[-1]:.4f})")
axes[1, 0].legend()
axes[1, 0].grid(True, alpha=0.3)

# Portfolio vs Option (Weekly)
axes[1, 1].plot(times, port_weekly, label='Portfolio')
axes[1, 1].plot(times, opt_val, '--', alpha=0.7, label='Option Value')
axes[1, 1].set_xlabel('Time (Years)')
axes[1, 1].set_ylabel('Value')
axes[1, 1].set_title(f"Weekly Hedging (Error: {port_weekly[-1] - opt_val[-1]:.4f})")
axes[1, 1].legend()
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Volatility mismatch: Portfolio evolution with different sigma_stock (daily vs weekly hedging)
np.random.seed(42)

fig, ax = plt.subplots(figsize=(12, 8))

sigma_combinations = [(0.15, 0.20), (0.20, 0.20), (0.25, 0.20)]

for (sigma_stock, sigma_hedge) in sigma_combinations:
    path = euler_path(S0, r, sigma_stock, T, M)
    port_daily, _, _ = simulate_portfolio_with_path(path, K, T, r, sigma_hedge, 1)
    port_weekly, _, _ = simulate_portfolio_with_path(path, K, T, r, sigma_hedge, 5)
    label_daily = f'sigma_stock={sigma_stock*100:.1f}% (daily)'
    label_weekly = f'sigma_stock={sigma_stock*100:.1f}% (weekly)'
    ax.plot(times, port_daily, label=label_daily, linestyle='-')
    ax.plot(times, port_weekly, label=label_weekly, linestyle='--')

ax.set_xlabel("Time (Years)")
ax.set_ylabel("Portfolio Value")
ax.set_title("Volatility Mismatch Impact (Daily vs Weekly Hedging)")
ax.legend()
ax.grid(True, alpha=0.3)
plt.show()